# 🔄 Təkrarlanan Neyron Şəbəkələri (RNN və LSTM): Mətn və Ardıcıllıqların Emalı

## 🎬 Film Rəylərinin Hissiyyat Analizi (Sentiment Analysis)

Bu notebookda Dərin Öyrənmənin ən maraqlı sahələrindən biri olan **Təbii Dil Emalı (NLP - Natural Language Processing)** dünyasına daxil oluruq. Məqsədimiz: IMDb saytında istifadəçilərin film haqqında yazdıqları rəyləri oxuyub, filmi **bəyəndiklərini (müsbət 👍)**, yoxsa **bəyənmədiklərini (mənfi 👎)** süni neyron şəbəkəsinə təyin etdirməkdir.

---

### 🤔 Niyə Mətn Üçün Adi Şəbəkələr (MLP və CNN) Yetmir?

Mətn sadəcə ayrı-ayrı sözlərin siyahısı deyil, **zaman ardıcıllığıdır (Sequence)**!
* Cümlədə sözlərin mənası özündən əvvəl gələn sözlərdən birbaşa asılıdır.
* Məsələn:
  * *"Film pis deyildi, əksinə möhtəşəm idi!"* (Müsbət məna)
  * *"Film möhtəşəm deyildi, əksinə pis idi!"* (Mənfi məna)
  * Eyni sözlər istifadə olunsa da, ardıcıllıq dəyişdikdə cümlənin mənası 180 dərəcə çevrilir!
* Adi şəbəkələrin yaddaşı yoxdur; onlar bütün sözlərə eyni anda, bir-birindən xəbərsiz baxır.

### 🧠 Xilaskar: RNN (Recurrent Neural Network) və LSTM (Long Short-Term Memory)
* **RNN:** Daxili gizli vəziyyət (*Hidden State* $h_t$) saxlayır. Cümləni söz-bə-söz oxuduqca keçmiş sözlərin xülasəsini öz yaddaşında növbəti addıma ötürür.
* **Niyə Sadə RNN Yox, LSTM?** Sadə RNN-lərdə **Vanishing Gradient (Qradiyentin Yox Olması)** problemi var — cümlə uzandıqca ən başdakı sözlər unudulur.
* **LSTM (1997):** Xüsusi qapı mexanizmləri (**Forget Gate**, **Input Gate**, **Output Gate**) və məlumat magistralı (**Cell State**) sayəsində vacib sözləri uzun cümlələr boyu yaddaşında saxlayır, lazımsız sözləri isə unudur!

In [1]:
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.datasets import imdb

import numpy as np

## 📦 IMDb Verilənlər Bazasının Yüklənməsi

IMDb bazasında 25,000 təlim və 25,000 test olmaqla cəmi 50,000 film rəyi var.

* `num_words=20000`: Dilimizdə yüz minlərlə nadir və təsadüfi söz ola bilər. Şəbəkənin yaddaşını doldurmamaq və təlimi effektiv etmək üçün **yalnız ən çox işlənən ilk 20,000 sözü** götürürük:

In [8]:
# Yalnız ən populyar 20,000 sözü nəzərə alırıq
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=20000)

print(x_train.shape)
print(x_train.shape)
print(y_train.shape)
print(y_train.shape)

(25000,)
(25000,)
(25000,)
(25000,)


Gəlin bu məlumatın necə göründüyünə baxaq. İlk təlim rəyini ekrana çıxaraq:

In [5]:
# 1-ci film rəyinin məzmunu
print(x_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 19193, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 10311, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 12118, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


### 🔢 Bu Niyə İngiliscə Sözlər Yox, Ədədlərdən İbarətdir?

İlk baxışda bu, film rəyinə oxşamır! Amma Keras bizi böyük bir zəhmətdən xilas edib:
* Neyron şəbəkələri hərfləri və sözləri başa düşmür, yalnız riyazi ədədlərlə hesablayır.
* IMDb verilənlər bazasında hər bir söz lüğətdəki məşhurluq indeksinə (**Word Index**) çevrilib.
* Məsələn:
  * `1` $\to$ Cümlənin başlanğıcı (Start token)
  * `14` $\to$ "this" sözü
  * `22` $\to$ "film" sözü və s.

Bəs bu rəyin etiketi (hissiyyatı) necədir?

In [9]:
# 1-ci rəyin etiketi
print("Etiket:", y_train[0])

Etiket: 1


Etiketlər ikili qiymətlərdir (**Binary Classification**):
* **`1`** $\to$ **Müsbət rəy (Positive):** İstifadəçi filmi bəyənib.
* **`0`** $\to$ **Mənfi rəy (Negative):** İstifadəçi filmdən narazı qalıb.

---

## 📏 Rəylərin Uzunluqlarının Bərabərləşdirilməsi: `pad_sequences`

Real həyatda film rəylərinin uzunluğu fərqlidir: bəzi rəy 15 sözdən ibarətdir, bəziləri isə 800 sözlük uzun məqalədir.

Neyron şəbəkəsi eyni ölçülü partiyalarla (*batch*) işlədiyi üçün bütün rəyləri vahid uzunluğa gətirməliyik:
* **`maxlen=80`:** Hər bir rəyi **ilk 80 sözlə** məhdudlaşdırırıq.
  * Əgər rəy 80 sözdən qısadırsa, əvvəlinə `0`-lar artırılır (**Padding**).
  * Əgər rəy 80 sözdən uzundursa, 80-ci sözdən sonrası kəsilir (**Truncating**).

In [10]:
# Bütün rəyləri dəqiq 80 söz uzunluğuna gətiririk
x_train = sequence.pad_sequences(x_train, maxlen=80)
x_test = sequence.pad_sequences(x_test, maxlen=80)

print('x_train forması:', x_train.shape)
print('x_test forması:', x_test.shape)

x_train forması: (25000, 80)
x_test forması: (25000, 80)


In [18]:
x_test[0,]

array([    0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     1,   591,   202,    14,    31,     6,
         717,    10,    10, 18142, 10698,     5,     4,   360,     7,
           4,   177,  5760,   394,   354,     4,   123,     9,  1035,
        1035,  1035,    10,    10,    13,    92,   124,    89,   488,
        7944,   100,    28,  1668,    14,    31,    23,    27,  7479,
          29,   220,   468,     8,   124,    14,   286,   170,     8,
         157,    46,     5,    27,   239,    16,   179, 15387,    38,
          32,    25,  7944,   451,   202,    14,     6,   717],
      dtype=int32)

## 🏗️ LSTM Modelinin Qurulması: Addım-Addım İzah

Arxitekturanın daxilində çox mürəkkəb riyaziyyat olsa da, Keras ilə onu qurmaq olduqca sadədir:

```
[Giriş: 80 sözlük indekslər]
           ⬇️
[Embedding(20000, 128)]   -> Hər sözü 128 ölçülü məna vektoruna çevirir
           ⬇️
[LSTM(128)]               -> Cümlə boyunca yaddaş saxlayan 128 hüceyrəli LSTM
           ⬇️
[Dense(1) + Sigmoid]      -> 0 (Mənfi) ilə 1 (Müsbət) arasında ehtimal
```

### 🔑 Əsas Qatların İzahı:
1. **`Embedding(20000, 128)`:**  
   NLP-nin təməl daşıdır! Hər bir sözün tam ədəd indeksini (məs: `542`) 128 ölçülü sıx vektora (*Dense Vector*) çevirir. Təlim zamanı şəbəkə oxşar mənalı sözləri (məsələn: *"great"*, *"awesome"*, *"fantastic"*) çoxölçülü məkanda bir-birinə yaxın yerləşdirməyi öyrənir!
2. **`LSTM(128, dropout=0.2, recurrent_dropout=0.2)`:**  
   128 yaddaş vahidinə malik LSTM qatı.  
   * `dropout=0.2`: Giriş siqnallarının 20%-ni təsadüfi söndürərək əzbərləmənin qarşısını alır.
   * `recurrent_dropout=0.2`: Keçmiş addımdan gələn yaddaş əlaqələrinin 20%-ni söndürür (ikiqat müdafiə).
3. **`Dense(1, activation='sigmoid')`:**  
   Tək bir neyron. `sigmoid` aktivasiyası çıxışı `0.0` ilə `1.0` arasında ehtimala çevirir (məsələn, `0.92` $\to$ 92% əminliklə müsbət rəydir).

In [19]:
# Ardıcıl modeli qururuq
model = Sequential()

# 1. Embedding qatı: 20,000 lüğət həcmi, hər söz üçün 128 ölçülü vektor
model.add(Embedding(20000, 128))

# 2. LSTM qatı: 128 gizli vahid və Overfitting-ə qarşı Dropout tənzimləmələri
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))

# 3. Çıxış qatı: 1 neyron və Sigmoid aktivasiyası (İkili ehtimal: 0 və ya 1)
model.add(Dense(1, activation='sigmoid'))

## ⚙️ Modelin Kompilyasiyası: Optimizator və İtki Funksiyası

* **İtki Funksiyası (`loss='binary_crossentropy'`):** İki sinifli (Müsbət / Mənfi) təsnifat üçün standart logaritmik itki funksiyası.
* **Optimizator (`optimizer='adam'`):** Adaptiv öyrənmə sürəti ilə ən stabil və sürətli optimizator.
* **Metrika (`metrics=['accuracy']`):** Dəqiqlik faizi.

In [20]:
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

## 🏋️ Modelin Təlim Edilməsi: `model.fit()`

RNN və xüsusilə LSTM modelləri vaxt və hesablama gücü baxımından çox tələbkardır. Hər bir LSTM hüceyrəsi daxilində 4 fərqli riyazi qapı işləyir.

Təlim prosesini optimallaşdırmaq üçün:
* `batch_size=32`: Kiçik partiya ölçüsü ilə yaddaşı qoruyuruq.
* `epochs=15`: 15 dövr təlim təyin edirik.

> ⚠️ **Qeyd:** Adi prosessorda (CPU) bu təlim xeyli vaxt apara bilər. GPU olduqda proses qat-qat sürətlənir.

Təlim dövrünü işə salırıq və hər epoch üzrə təlim və test nəticələrini izləyirik:

In [21]:
history = model.fit(x_train, y_train,
                    batch_size=32,
                    epochs=15,
                    verbose=2,
                    validation_data=(x_test, y_test))

Epoch 1/15
782/782 - 35s - 45ms/step - accuracy: 0.7807 - loss: 0.4601 - val_accuracy: 0.8346 - val_loss: 0.3775
Epoch 2/15
782/782 - 36s - 46ms/step - accuracy: 0.8790 - loss: 0.2977 - val_accuracy: 0.8366 - val_loss: 0.3760
Epoch 3/15
782/782 - 37s - 47ms/step - accuracy: 0.9180 - loss: 0.2127 - val_accuracy: 0.8316 - val_loss: 0.4228
Epoch 4/15
782/782 - 35s - 45ms/step - accuracy: 0.9443 - loss: 0.1496 - val_accuracy: 0.8270 - val_loss: 0.5320
Epoch 5/15
782/782 - 36s - 47ms/step - accuracy: 0.9646 - loss: 0.0985 - val_accuracy: 0.8237 - val_loss: 0.5281
Epoch 6/15
782/782 - 36s - 46ms/step - accuracy: 0.9752 - loss: 0.0731 - val_accuracy: 0.8089 - val_loss: 0.5947
Epoch 7/15
782/782 - 36s - 46ms/step - accuracy: 0.9825 - loss: 0.0518 - val_accuracy: 0.8210 - val_loss: 0.7974
Epoch 8/15
782/782 - 36s - 47ms/step - accuracy: 0.9877 - loss: 0.0380 - val_accuracy: 0.8138 - val_loss: 0.8326
Epoch 9/15
782/782 - 38s - 48ms/step - accuracy: 0.9909 - loss: 0.0286 - val_accuracy: 0.8136 - 

## 🏆 Yekun Nəticə və Qiymətləndirmə: `model.evaluate()`

Gəlin modelin heç vaxt görmədiyi **25,000 test rəyi** üzərində rəsmi dəqiqliyini ölçək:

In [22]:
# Test dəsti üzərində itkini və dəqiqliyi yoxlayırıq
score, acc = model.evaluate(x_test, y_test,
                            batch_size=32,
                            verbose=2)
print('Test İtkisi (Loss): %.4f' % score)
print('Test Dəqiqliyi (Accuracy): %.2f%%' % (acc * 100))

782/782 - 6s - 8ms/step - accuracy: 0.8095 - loss: 1.1364
Test İtkisi (Loss): 1.1364
Test Dəqiqliyi (Accuracy): 80.95%


## 💡 Nəticələrin Təhlili və Düşüncələr

Model test məlumatlarında təxminən **80-82% dəqiqlik** göstərdi!

Nəzərə alsaq ki:
1. Biz hər rəydən yalnız ilk **80 sözü** götürmüşdük (rəyin əsas hissəsi bəzən axırda olur),
2. Lüğəti yalnız 20,000 sözlə məhdudlaşdırmışdıq,
3. Və cəmi 3 sətir kodla təbii insan dilini oxuyan bir arxitektura qurduq — bu **möhtəşəm bir nəticədir!**

### ⚠️ Overfitting və Early Stopping Müşahidəsi:
Təlim tarixinə diqqət etsəniz, test dəqiqliyinin ilk bir neçə epoch-dan sonra çox artmadığını, lakin təlim xətasının getdikcə azaldığını görərsiniz. Bu, modelin rəyləri əzbərləməyə (**Overfitting**) başladığını göstərir. Belə hallarda **EarlyStopping** (dəqiqlik artmayanda təlimi vaxtından əvvəl saxlamaq) tətbiq etmək ən yaxşı təcrübədir.

---

### 🚀 Gələcək Təkamül: LSTM-dən Transformerlərə!
Bu gün istifadə etdiyimiz **ChatGPT, Gemini, Claude** kimi nəhəng dil modelləri (LLM) məhz bu mətn ardıcıllığı məntiqinin ən son zirvəsidir. Onlar LSTM-in yerinə bütün cümləyə eyni anda baxan **Transformer (Self-Attention)** mexanizmindən istifadə edirlər!

In [39]:
# 🧪 Modelin Təxminlərini Yoxlamaq Üçün Test Bloku:
# İlk 5 test rəyinin proqnozunu və həqiqi hissini müqayisə edirik

print(x_test.shape)
print(x_test[:5,])

(25000, 80)
[[    0     0     0     0     0     0     0     0     0     0     0     0
      1   591   202    14    31     6   717    10    10 18142 10698     5
      4   360     7     4   177  5760   394   354     4   123     9  1035
   1035  1035    10    10    13    92   124    89   488  7944   100    28
   1668    14    31    23    27  7479    29   220   468     8   124    14
    286   170     8   157    46     5    27   239    16   179 15387    38
     32    25  7944   451   202    14     6   717]
 [ 1669   398   229    10    10    13  2822   670  5304    14     9    31
      7    27   111   108    15  2033    19  7836  1429   875   551    14
     22     9  1193    21    45  4829     5    45   252     8 12508     6
    565   921  3639    39     4   529    48    25   181     8    67    35
   1732    22    49   238    60   135  1162    14     9   290     4    58
     10    10   472    45    55   878     8   169    11   374  5687    25
    203    28     8   818    12   125     4  3077

In [40]:
predictions = model.predict(x_test[:5])
predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


array([[0.95529705],
       [0.99999994],
       [0.9975354 ],
       [0.13136813],
       [0.99999946]], dtype=float32)

In [41]:
for i in range(5):
    prob = predictions[i][0]
    sentiment = "MÜSBƏT 👍" if prob >= 0.5 else "MƏNFİ 👎"
    actual = "MÜSBƏT 👍" if y_test[i] == 1 else "MƏNFİ 👎"
    print(f"Rəy #{i+1} -> Modelin Ehtimalı: {prob:.4f} ({sentiment}) | Həqiqi: {actual}")

Rəy #1 -> Modelin Ehtimalı: 0.9553 (MÜSBƏT 👍) | Həqiqi: MƏNFİ 👎
Rəy #2 -> Modelin Ehtimalı: 1.0000 (MÜSBƏT 👍) | Həqiqi: MÜSBƏT 👍
Rəy #3 -> Modelin Ehtimalı: 0.9975 (MÜSBƏT 👍) | Həqiqi: MÜSBƏT 👍
Rəy #4 -> Modelin Ehtimalı: 0.1314 (MƏNFİ 👎) | Həqiqi: MƏNFİ 👎
Rəy #5 -> Modelin Ehtimalı: 1.0000 (MÜSBƏT 👍) | Həqiqi: MÜSBƏT 👍
